In [1]:
from bioturing_spatialx._io import SpatialData
sdata = SpatialData("/nfsshared/personal/tin.tra_bioturing.com/spatial_study/study/ST-01KT6BDY3XEJHKQ9X5CRP5TDAT/spatial/SP-01KT6BDYQXQ0HJZG9KH4X49V3R")

src_img = sdata.images["test cosmx_images (1)"]["0"][:]

/home/toanvong/miniconda3/envs/spatialx-general/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/home/toanvong/spatialx/pyapps/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/toanvong/spatialx/pyapps/lib/python3.12/site-packages/zarr/core/sync.py:119: ZarrRuntimeWarning: cache_attrs is not yet implemented
  return await coro


In [2]:
import matplotlib.pyplot as plt

from skimage.transform import resize

def plot_img(img, scale=1):
    if img.ndim == 3 and img.shape[0] == 1:
        img = img[0]
    h, w = img.shape[:2]
    new_h, new_w = int(h * scale), int(w * scale)
    print(f"{(new_h, new_w)=}")
    img_small = resize(img, (new_h, new_w), preserve_range=True).astype(img.dtype)
    print(f"{(new_h, new_w)=}")
    plt.figure(figsize=(12, 8))
    plt.imshow(img_small, cmap='gray')
    plt.axis('off')
    plt.colorbar()
    plt.show()

In [3]:
src_img

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(1, 13921, 42636), dtype=uint16)

In [4]:
from pathlib import Path
import json

TRANSFORM_PATH = Path("/home/toanvong/spatialx/__scratch/alignment-params-test-cosmx_images.json")

with open(TRANSFORM_PATH, "r") as f:
    data = json.load(f)
    print(data)

{'reference_image': {'image_id': '1780477267_b36af349f949465f95666fcca7c027f1', 'alignment_id': 'aln_536506970690b53cde529c09f8cd27c1'}, 'alignment': {'transformation': {'transformation_type': 'composed', 'params': {'transforms': [{'transformation_type': 'tps', 'params': {'affine_params': {'A': [[-0.04101859347567449, 1.1354183686407158], [0.14383934594271244, 0.8537488549729029]], 'b': [-15844.93593553328, -23154.54029431603]}, 'weights_x': [8.769721709443136e-05, 9.579634884933404e-05, -1.1603389723547491e-05, 0.0003412051556302696, -0.000463879414325662, -4.921591752482552e-05], 'weights_y': [-4.385274188823425e-06, -2.083150988749989e-05, -2.500369070291277e-05, -2.2877762320048085e-05, 4.761755747591026e-05, 2.548067962337391e-05], 'control_points': [3712.913128601038, 7563.413933952008, 15449.637709533676, 10939.73196408332, 29437.240977220517, 5634.0893453055505, 19790.618033988216, 4026.318854766836, 14163.4213171027, 5553.700820778617, 33456.66720356731, 9010.407375436855]}}, 

In [5]:
from bioturing_spatialx.alignment import Alignment

alignment = Alignment.model_validate(data["alignment"])
alignment.transformation.inverse((0, 0))

Point(point=[63629.112373628166, -8997.142198279022])

In [6]:
import numpy as np
from scipy import ndimage


def _tps_kernel_vec(r):
    result = np.zeros_like(r)
    mask = r > 1e-10
    result[mask] = r[mask] ** 2 * np.log(r[mask])
    return result


def _apply_tps_vec(x, y, A, b, wx, wy, cpx, cpy):
    n = len(wx)
    fx = A[0, 0] * x + A[0, 1] * y + b[0]
    fy = A[1, 0] * x + A[1, 1] * y + b[1]
    for i in range(n):
        dx = x - cpx[i]
        dy = y - cpy[i]
        r = np.sqrt(dx**2 + dy**2)
        U = _tps_kernel_vec(r)
        fx += wx[i] * U
        fy += wy[i] * U
    return fx, fy


def _tps_jacobian_vec(x, y, A, wx, wy, cpx, cpy):
    n = len(wx)
    dxdx = np.full_like(x, A[0, 0])
    dxdy = np.full_like(x, A[0, 1])
    dydx = np.full_like(x, A[1, 0])
    dydy = np.full_like(x, A[1, 1])
    for i in range(n):
        dx = x - cpx[i]
        dy = y - cpy[i]
        r = np.sqrt(dx**2 + dy**2)
        mask = r > 1e-10
        factor = np.zeros_like(r)
        factor[mask] = 2.0 * np.log(r[mask]) + 1.0
        dU_dx = factor * dx
        dU_dy = factor * dy
        dxdx += wx[i] * dU_dx
        dxdy += wx[i] * dU_dy
        dydx += wy[i] * dU_dx
        dydy += wy[i] * dU_dy
    return dxdx, dxdy, dydx, dydy


def _invert_tps_vec(tx, ty, A, b, wx, wy, cpx, cpy, max_iter=20, tol=1e-6):
    x = tx.copy()
    y = ty.copy()
    for _ in range(max_iter):
        fx, fy = _apply_tps_vec(x, y, A, b, wx, wy, cpx, cpy)
        dxdx, dxdy, dydx, dydy = _tps_jacobian_vec(x, y, A, wx, wy, cpx, cpy)
        rx = fx - tx
        ry = fy - ty
        det = dxdx * dydy - dxdy * dydx
        ok = np.abs(det) > 1e-10
        dx = np.zeros_like(x)
        dy = np.zeros_like(y)
        dx[ok] = (-rx[ok] * dydy[ok] + ry[ok] * dxdy[ok]) / det[ok]
        dy[ok] = (rx[ok] * dydx[ok] - ry[ok] * dxdx[ok]) / det[ok]
        x += dx
        y += dy
        if np.max(np.sqrt(dx**2 + dy**2)) < tol:
            break
    return x, y


def _invert_affine_vec(tx, ty, A, b):
    det = A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]
    inv_det = 1.0 / det
    iA00 = A[1, 1] * inv_det
    iA01 = -A[0, 1] * inv_det
    iA10 = -A[1, 0] * inv_det
    iA11 = A[0, 0] * inv_det
    dx = tx - b[0]
    dy = ty - b[1]
    return iA00 * dx + iA01 * dy, iA10 * dx + iA11 * dy


def _extract_composed_params(transform):
    transforms = transform.params.transforms
    tps = transforms[0]
    affine = transforms[1]
    tA = np.array(affine.params.A, dtype=np.float64)
    tb = np.array(affine.params.b, dtype=np.float64)
    tpA = np.array(tps.params.affine_params.A, dtype=np.float64)
    tpb = np.array(tps.params.affine_params.b, dtype=np.float64)
    wx = np.array(tps.params.weights_x, dtype=np.float64)
    wy = np.array(tps.params.weights_y, dtype=np.float64)
    cp = tps.params.control_points.lst
    cpx = np.array(cp[0::2], dtype=np.float64)
    cpy = np.array(cp[1::2], dtype=np.float64)
    return tA, tb, tpA, tpb, wx, wy, cpx, cpy


def _inverse_composed_vec(tx, ty, tA, tb, tpA, tpb, wx, wy, cpx, cpy):
    ax, ay = _invert_affine_vec(tx, ty, tA, tb)
    sx, sy = _invert_tps_vec(ax, ay, tpA, tpb, wx, wy, cpx, cpy)
    return sx, sy


def transform_image(src_img, transform, output_shape=None, output_extent=None, pixel_scale=None):
    """
    Transform an image using inverse wrapping.

    For each pixel in the output (target) image, find the corresponding
    source pixel via the inverse of the transformation, then sample the
    source image with bilinear interpolation.

    Parameters
    ----------
    src_img : np.ndarray
        Source image, shape (C, H, W) or (H, W)
    transform : AlignmentTransformation
        Composed [TPS, Affine] transformation with .inverse() method
    output_shape : tuple (out_h, out_w), optional
        Output image dimensions. If None, same as source.
    output_extent : tuple (x_min, y_min, x_max, y_max), optional
        Target-space coordinate bounds for the output image.
        If None, determined by forward-transforming source image corners.
    pixel_scale : float or tuple (sy, sx), optional
        Scale from pixel coordinates to transformation coordinates.
        E.g. pixel_scale=32 means each pixel maps to 32 units in transform space.

    Returns
    -------
    np.ndarray
        Warped image with same number of channels as src_img
    """
    squeeze = False
    if src_img.ndim == 2:
        src_img = src_img[np.newaxis]
        squeeze = True

    C, H, W = src_img.shape
    if pixel_scale is None:
        sy, sx = 1.0, 1.0
    elif isinstance(pixel_scale, (int, float)):
        sy, sx = float(pixel_scale), float(pixel_scale)
    else:
        sy, sx = float(pixel_scale[0]), float(pixel_scale[1])

    if output_shape is None:
        output_shape = (H, W)
    out_h, out_w = output_shape

    if output_extent is None:
        corners_px = [(0, 0), (W - 1, 0), (0, H - 1), (W - 1, H - 1)]
        corners_world = [(cx * sx, cy * sy) for cx, cy in corners_px]
        mapped = [transform.transform(c) for c in corners_world]
        xs = [p.x for p in mapped]
        ys = [p.y for p in mapped]
        output_extent = (min(xs), min(ys), max(xs), max(ys))

    x_min, y_min, x_max, y_max = output_extent

    xs = np.linspace(x_min, x_max, out_w)
    ys = np.linspace(y_min, y_max, out_h)
    xx, yy = np.meshgrid(xs, ys)

    tA, tb, tpA, tpb, wx, wy, cpx, cpy = _extract_composed_params(transform)
    src_x, src_y = _inverse_composed_vec(
        xx.ravel(), yy.ravel(), tA, tb, tpA, tpb, wx, wy, cpx, cpy
    )
    src_x = src_x.reshape(out_h, out_w)
    src_y = src_y.reshape(out_h, out_w)

    sample_col = src_x / sx
    sample_row = src_y / sy

    result = np.zeros((C, out_h, out_w), dtype=src_img.dtype)
    coords = np.array([sample_row, sample_col])
    for c in range(C):
        result[c] = ndimage.map_coordinates(
            src_img[c], coords, order=1, mode='constant', cval=0
        )

    if squeeze:
        return result[0]
    return result

In [7]:
def check_points_in_image(points, img_shape, pixel_scale=None):
    """
    Check which reference points reside inside the image bounds.

    Parameters
    ----------
    points : list of (x, y) or flat [x0, y0, x1, y1, ...]
        Points in world/transform coordinates.
    img_shape : tuple (C, H, W) or (H, W)
        Image dimensions.
    pixel_scale : float or (sy, sx), optional
        Scale from pixel coordinates to world coordinates.

    Returns
    -------
    dict with 'inside' and 'outside' lists of (index, (x, y))
    """
    if len(img_shape) == 3:
        _, H, W = img_shape
    else:
        H, W = img_shape

    if pixel_scale is None:
        sy, sx = 1.0, 1.0
    elif isinstance(pixel_scale, (int, float)):
        sy, sx = float(pixel_scale), float(pixel_scale)
    else:
        sy, sx = float(pixel_scale[0]), float(pixel_scale[1])

    max_x, max_y = W * sx, H * sy

    if isinstance(points, (list, np.ndarray)) and len(points) > 0 and not isinstance(points[0], (list, tuple)):
        pts = [(points[i], points[i + 1]) for i in range(0, len(points), 2)]
    else:
        pts = list(points)

    inside, outside = [], []
    for i, (x, y) in enumerate(pts):
        (inside if 0 <= x <= max_x and 0 <= y <= max_y else outside).append((i, (x, y)))

    return {"inside": inside, "outside": outside}

In [8]:
kp = data["alignment"]["preparams"]["keypoints"]

print("Source keypoints vs src_img bounds:")
result = check_points_in_image(kp[0], src_img.shape, pixel_scale=(31.9289, 31.9850))
print(f"  Inside:  {len(result['inside'])}/{len(result['inside']) + len(result['outside'])}")
for idx, pt in result['outside']:
    print(f"  Outside: point {idx} = ({pt[0]:.1f}, {pt[1]:.1f})")

print("\nTarget keypoints vs src_img bounds:")
result = check_points_in_image(kp[1], src_img.shape, pixel_scale=(31.9289, 31.9850))
print(f"  Inside:  {len(result['inside'])}/{len(result['inside']) + len(result['outside'])}")
for idx, pt in result['outside']:
    print(f"  Outside: point {idx} = ({pt[0]:.1f}, {pt[1]:.1f})")

Source keypoints vs src_img bounds:
  Inside:  6/6

Target keypoints vs src_img bounds:
  Inside:  6/6


In [ ]:
warped = transform_image(
    src_img,
    alignment.transformation,
    pixel_scale=(31.9289, 31.9850),
)
plot_img(warped)